# Get HUC10 (and HUC12-in-HUC10) shapes for ConvLSTM

## What this notebook does

1. **Defines which watersheds we care about** — a list of HUC8 IDs (e.g. 17020009, 17110005) and which HUC10s to exclude (e.g. Canadian ones with missing data).
2. **Authenticates with Google Earth Engine** — needed because shapes come from the USGS Watershed Boundary Dataset hosted on Earth Engine.
3. **For each HUC8**, calls `get_geos_with_name(huc8, "10")` to fetch all HUC10 polygons inside that HUC8 (with names).
4. **For each HUC10**, saves one GeoJSON file containing that single watershed’s boundary and uploads it to S3.
5. **For each HUC10**, also fetches **all HUC12 polygons that lie inside it**, saves them as a single GeoJSON (`Huc12_in_<HUC10>.geojson`), and uploads that to S3.
6. **Builds metadata CSVs**:
   - `huc10_list.csv` (HUC8_ID, HUC10_ID, HUC10_Name) so downstream code knows the list of HUC10s and their parent HUC8.
   - `huc12_in_huc10_list.csv` (HUC10_ID, HUC12_ID, HUC12_Name) so downstream code and analysis can link each HUC12 back to its HUC10.

**Output on S3 (bucket `convlstm-model-ready`):**
- `huc10_shapes/Huc10_in_<id>.geojson` — one file per HUC10.
- `huc12_shapes_in_huc10/Huc12_in_<huc10_id>.geojson` — one file per HUC10, containing all HUC12s inside that HUC10.
- `huc10_shapes/metadata/huc10_list.csv` — table of all HUC10s.
- `huc12_shapes_in_huc10/metadata/huc12_in_huc10_list.csv` — table linking each HUC12 (ID + name) to its parent HUC10.

**Run from repo root** (or ensure snowML is installed: `pip install -e .`). Configure AWS credentials (e.g. `aws configure`) for S3 upload.

In [1]:
# Standard library: paths and a temporary folder for writing files before uploading
import os
import tempfile

# Data and geometry
import pandas as pd
import geopandas as gpd

# S3 uploads
import boto3

# SnowML helper: fetches HUC boundaries from Google Earth Engine (USGS WBD dataset)
from snowML.datapipe.utils import get_geos

/Users/khaja/Documents/SnowML/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [5]:
# --- Which HUC8 regions to include (each HUC8 contains many HUC10 sub-watersheds) ---
HUC_MARITIME = [17020009, 17020011, 17110005, 17110006, 17110009]
HUC_MONTANE = [17010304, 17010302, 17060207, 17060208]
HUC_MIXED = [17030001, 17030002, 17110008]
HUC_EPHEMERAL = [17030003, 17020010, 17110007]
HUC_08_list = HUC_MARITIME + HUC_MIXED + HUC_EPHEMERAL + HUC_MONTANE

# HUC10s to skip (e.g. Canadian basins with missing SWE data)
EXCLUDED_HUC10S_CA = ["1711000501", "1711000502", "1711000503", "1711000504"]

# --- S3: bucket and key prefixes (folder paths in the bucket) ---
BUCKET = "convlstm-model-ready"
S3_PREFIX_SHAPES = "huc10_shapes"                 # HUC10 GeoJSON files go here
S3_PREFIX_HUC12_IN_HUC10 = "huc12_shapes_in_huc10"  # HUC12-in-HUC10 GeoJSON files go here
S3_PREFIX_METADATA = "huc10_shapes/metadata"       # CSV catalog goes here
s3_client = boto3.client("s3")

In [3]:
# Authenticate with Google Earth Engine (opens browser or prompts for login if needed)
get_geos.ee_creds()

True

In [ ]:
excluded_set = set(EXCLUDED_HUC10S_CA)  
rows = []  # HUC10 metadata rows
huc12_rows = []  # HUC12-in-HUC10 metadata rows

with tempfile.TemporaryDirectory() as tmpdir:
    # Loop over each parent HUC8 region
    for huc8 in HUC_08_list:
        # Fetch all HUC10 polygons (and names) inside this HUC8 from Earth Engine
        gdf = get_geos.get_geos_with_name(huc8, "10", s3_save=False)
        # Drop any HUC10s that are in the exclusion list (e.g. Canadian)
        gdf = gdf[~gdf["huc_id"].astype(str).isin(excluded_set)]

        # One GeoJSON file per HUC10 (and companion HUC12-in-HUC10 file)
        for _, row in gdf.iterrows():
            huc10_id = row["huc_id"]
            # Single-row GeoDataFrame for this watershed’s boundary
            gdf_10 = gpd.GeoDataFrame([row], geometry="geometry", crs=gdf.crs)
            fname10 = f"Huc10_in_{huc10_id}.geojson"
            local_path10 = os.path.join(tmpdir, fname10)
            gdf_10.to_file(local_path10, driver="GeoJSON")
            # Upload to S3: huc10_shapes/Huc10_in_<id>.geojson
            s3_key10 = f"{S3_PREFIX_SHAPES}/{fname10}"
            s3_client.upload_file(local_path10, BUCKET, s3_key10)

            # For this HUC10, also fetch all HUC12s that lie inside it and save as one GeoJSON
            gdf12 = get_geos.get_geos_with_name(huc10_id, "12", s3_save=False)
            fname12 = f"Huc12_in_{huc10_id}.geojson"
            local_path12 = os.path.join(tmpdir, fname12)
            gdf12.to_file(local_path12, driver="GeoJSON")
            # Upload to S3: huc12_shapes_in_huc10/Huc12_in_<huc10_id>.geojson
            s3_key12 = f"{S3_PREFIX_HUC12_IN_HUC10}/{fname12}"
            s3_client.upload_file(local_path12, BUCKET, s3_key12)

            # Collect metadata for the HUC10 CSV
            rows.append({
                "HUC8_ID": huc8,
                "HUC10_ID": huc10_id,
                "HUC10_Name": row["huc_name"],
            })
            # Collect metadata for the HUC12-in-HUC10 CSV
            for _, row12 in gdf12.iterrows():
                huc12_rows.append({
                    "HUC10_ID": huc10_id,
                    "HUC12_ID": row12["huc_id"],
                    "HUC12_Name": row12.get("huc_name", ""),
                })
        print(f"HUC8 {huc8}: uploaded {len(gdf)} HUC10 shapes and HUC12-in-HUC10 shapes")

    # Build and upload the HUC10 catalog CSV (one row per HUC10)
    huc10_table = pd.DataFrame(rows)
    csv_path = os.path.join(tmpdir, "huc10_list.csv")
    huc10_table.to_csv(csv_path, index=False)
    s3_client.upload_file(csv_path, BUCKET, f"{S3_PREFIX_METADATA}/huc10_list.csv")
    print(f"Uploaded s3://{BUCKET}/{S3_PREFIX_METADATA}/huc10_list.csv ({len(huc10_table)} HUC10s)")

    # Build and upload the HUC12-in-HUC10 catalog CSV (one row per HUC12)
    huc12_table = pd.DataFrame(huc12_rows)
    csv_path12 = os.path.join(tmpdir, "huc12_in_huc10_list.csv")
    huc12_table.to_csv(csv_path12, index=False)
    s3_client.upload_file(csv_path12, BUCKET, f"{S3_PREFIX_HUC12_IN_HUC10}/metadata/huc12_in_huc10_list.csv")
    print(f"Uploaded s3://{BUCKET}/{S3_PREFIX_HUC12_IN_HUC10}/metadata/huc12_in_huc10_list.csv ({len(huc12_table)} HUC12s)")

HUC8 17020009: uploaded 3 HUC10 shapes and HUC12-in-HUC10 shapes
HUC8 17020011: uploaded 7 HUC10 shapes and HUC12-in-HUC10 shapes
HUC8 17110005: uploaded 7 HUC10 shapes and HUC12-in-HUC10 shapes
HUC8 17110006: uploaded 4 HUC10 shapes and HUC12-in-HUC10 shapes
HUC8 17110009: uploaded 7 HUC10 shapes and HUC12-in-HUC10 shapes
HUC8 17030001: uploaded 7 HUC10 shapes and HUC12-in-HUC10 shapes
HUC8 17030002: uploaded 3 HUC10 shapes and HUC12-in-HUC10 shapes
HUC8 17110008: uploaded 3 HUC10 shapes and HUC12-in-HUC10 shapes
HUC8 17030003: uploaded 12 HUC10 shapes and HUC12-in-HUC10 shapes
HUC8 17020010: uploaded 5 HUC10 shapes and HUC12-in-HUC10 shapes
HUC8 17110007: uploaded 2 HUC10 shapes and HUC12-in-HUC10 shapes
HUC8 17010304: uploaded 11 HUC10 shapes and HUC12-in-HUC10 shapes
HUC8 17010302: uploaded 3 HUC10 shapes and HUC12-in-HUC10 shapes
HUC8 17060207: uploaded 11 HUC10 shapes and HUC12-in-HUC10 shapes
HUC8 17060208: uploaded 6 HUC10 shapes and HUC12-in-HUC10 shapes
Uploaded s3://convlstm